# Deep Dive: Agentic Retrieval Augmented Generation

Extended from the [Agentic Router notebook](https://github.com/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/001.%20Agentic%20Router.ipynb).

**Extensions implemented:**
- **Part 1 (required):** Sub-query division — compound questions split into independent sub-queries, each routed and answered concurrently, results synthesised with namespaced citations.
- **Bonus:** RBAC-aware semantic caching — FAISS-backed tagged cache, preventing information leakage across access levels.

## Setup and Dependencies

In [1]:
!pip install -q openai qdrant-client transformers==4.48.0 tavily-python faiss-cpu torch python-dotenv nest_asyncio ipywidgets

In [2]:
import os
import re
import json
import time
import asyncio
import numpy as np

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import faiss
import nest_asyncio
from openai import OpenAI, OpenAIError
from transformers import AutoTokenizer, AutoModel
import qdrant_client
from tavily import TavilyClient

nest_asyncio.apply()

## API Keys

Copy `.env-example` to `.env` and fill in your keys:

```bash
cp .env-example .env
```

Or export them as environment variables.

In [3]:
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.environ.get("OPENAI_API_KEY", "")
tavily_api_key = os.environ.get("TAVILY_API_KEY", "")

openaiclient = OpenAI(api_key=openai_api_key)
tavily_client = TavilyClient(api_key=tavily_api_key)

MODEL = "gpt-4o"

## 1. Internet Tool (Tavily)

In [4]:
def get_internet_content(user_query: str, action: str) -> str:
    print("Getting your response from the internet 🌐 ...")
    try:
        data = tavily_client.search(query=user_query, max_results=5)
        results = data.get("results", [])
        if not results:
            return "No results found."
        parts = []
        for i, result in enumerate(results, start=1):
            title = result.get("title", "")
            content = result.get("content", "")
            url = result.get("url", "")
            if content:
                parts.append(f"[{i}] {title}\n    {content}\n    Source: {url}")
        return "\n\n".join(parts) if parts else "No results found."
    except Exception as err:
        return f"Search error: {err}"

## 2. Router Query Function

- `OPENAI_QUERY` — OpenAI docs, agents, APIs
- `10K_DOCUMENT_QUERY` — Lyft/Uber SEC 10-K filings
- `INTERNET_QUERY` — anything else (live web)

In [5]:
def route_query(user_query: str) -> dict:
    router_system_prompt = f"""
    As a professional query router, classify user input into one of three categories:
    1. "OPENAI_QUERY": Questions about OpenAI agents, models, APIs, guardrails, embeddings.
    2. "10K_DOCUMENT_QUERY": Questions about Lyft or Uber SEC 10-K annual reports.
    3. "INTERNET_QUERY": Everything else requiring real-time or broader web information.

    Always respond in valid JSON:
    {{
        "action": "OPENAI_QUERY" | "10K_DOCUMENT_QUERY" | "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words. Leave empty if INTERNET_QUERY"
    }}

    User: {user_query}
    """
    try:
        response = openaiclient.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": router_system_prompt}]
        )
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        return json.loads(json_match.group())
    except (OpenAIError, json.JSONDecodeError, AttributeError) as err:
        return {"action": "INTERNET_QUERY", "reason": str(err), "answer": ""}

## 3. Qdrant Vector Database

The pre-built Qdrant collections (10-K filings + OpenAI docs) ship with the course repo.
If `Agentic_RAG/qdrant_data` is missing (Colab or local), only that folder is fetched from the course repo via a sparse clone (~16 MB).

In [6]:
import shutil
import subprocess
import tempfile

_REPO = "https://github.com/hamzafarooq/multi-agent-course.git"
_DATA_IN_REPO = "modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data"

try:
    import google.colab
    BASE_DIR = "/content"
except ImportError:
    BASE_DIR = os.getcwd()

QDRANT_PATH = os.path.join(BASE_DIR, "Agentic_RAG", "qdrant_data")


def ensure_qdrant_data(path: str) -> None:
    """Sparse-clone only the Qdrant folder from the course repo if collections are missing."""
    if os.path.isdir(os.path.join(path, "collection")):
        return
    print("Downloading Qdrant data from the course repo...")
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", _REPO, tmp],
            check=True,
        )
        subprocess.run(["git", "-C", tmp, "sparse-checkout", "set", _DATA_IN_REPO], check=True)
        shutil.rmtree(path, ignore_errors=True)  # drop an empty store left by a previous run
        shutil.copytree(os.path.join(tmp, _DATA_IN_REPO), path)


ensure_qdrant_data(QDRANT_PATH)
print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /home/silviolleite/Projects/nootbook-assignment/Agentic_RAG/qdrant_data


## 4. Embedding Model

In [7]:
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)


def get_text_embeddings(text: str) -> np.ndarray:
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    outputs = text_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1)[0].detach().numpy()

<All keys matched successfully>


## 5. RAG Generator

In [8]:
def rag_formatted_response(user_query: str, context: list) -> str:
    rag_prompt = f"""
    Based on the given context, answer the user query: {user_query}
    Context:
    {context}
    Employ references to the ID of articles provided [ID].
    Referencing format: [1][2]...
    """
    response = openaiclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": rag_prompt}]
    )
    return response.choices[0].message.content


async def retrieve_and_response(user_query: str, action: str) -> str:
    collections = {
        "OPENAI_QUERY": "opnai_data",
        "10K_DOCUMENT_QUERY": "10k_data",
    }
    if action not in collections:
        return "Invalid action type for retrieval."
    try:
        query_vec = get_text_embeddings(user_query)
        hits = await client.query_points(
            collection_name=collections[action], query=query_vec, limit=3
        )
        contents = [p.payload["content"] for p in hits.points]
        if not contents:
            return "No relevant content found."
        return rag_formatted_response(user_query, contents)
    except Exception as err:
        return f"Retrieval error: {err}"

## 6. Agentic RAG Orchestrator

In [9]:
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}


def _execute_route(user_query: str, action: str) -> str:
    fn = routes.get(action)
    if not fn:
        return f"Unsupported action: {action}"
    if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
        return asyncio.run(fn(user_query, action))
    return fn(user_query, action)


def agentic_rag(user_query: str) -> None:
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")
    try:
        response = route_query(user_query)
    except Exception as e:
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\nRouting error: {e}\n")
        return
    action = response.get("action")
    print(f"{GREY}📍 Route: {action}\n📝 Reason: {response.get('reason')}\n⚙️  Processing...{RESET}\n")
    result = _execute_route(user_query, action)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{result}\n")

In [10]:
# agentic_rag("what was uber revenue in 2021?")
# agentic_rag("how to work with chat completions?")
# agentic_rag("List me new LLMs in 2025")

## 7. Role-Based Access Control (RBAC)

| User  | Role             | OPENAI_QUERY | 10K_DOCUMENT_QUERY | INTERNET_QUERY |
|-------|------------------|:---:|:---:|:---:|
| alice | engineer         | ✅  | ❌  | ✅  |
| bob   | finance_analyst  | ✅  | ✅  | ❌  |

In [11]:
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())

In [12]:
def secure_agentic_rag(user_id: str, user_query: str) -> str:
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )
    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    if role is None:
        msg = f"🚫 Access denied: unknown user '{user_id}'."
        print(f"{RED}{msg}{RESET}\n")
        return msg

    decision = route_query(user_query)
    action = decision.get("action")
    print(f"{GREY}📍 Route: {action}\n📝 Reason: {decision.get('reason')}{RESET}\n")

    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        msg = f"🚫 Access denied: role '{role}' cannot query {source}."
        print(f"{RED}{msg}{RESET}\n")
        return msg

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")
    result = _execute_route(user_query, action)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{result}\n")
    return result

In [13]:
# secure_agentic_rag("alice", "what was lyft revenue in 2022?")  # DENIED
# secure_agentic_rag("bob",   "what was lyft revenue in 2022?")  # ALLOWED

---
## Part 1 — Sub-Query Division

Compound questions are split into independent sub-questions. Sub-questions run **concurrently** via `asyncio.gather`. Citations are namespaced per sub-query (`[Q1:1]`, `[Q2:1]`) before synthesis, then an LLM composes a unified answer. A fallback (`_join_parts`) is used if the synthesis step drops citations.

| Input | Sub-queries | Routes |
|---|---|---|
| `"what was uber revenue in 2021?"` | 1 | `10K_DOCUMENT_QUERY` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 | both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and who won the 2022 FIFA World Cup?"` | 2 | `10K_DOCUMENT_QUERY` + `INTERNET_QUERY` |

In [14]:
def sub_queries(user_query: str) -> str:
    """Reference implementation — returns raw LLM response with a subQuestions JSON."""
    prompt = f"""
    You are a query router. If the input contains multiple distinct questions, break it into
    sub-questions. Otherwise, keep it as one. Return a JSON object like:

    {{"subQuestions": ["..."]}}

    Query: "{user_query}"
    Output:
    """
    response = openaiclient.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": prompt}]
    )
    return response.choices[0].message.content

In [15]:
_CITATION_RE = re.compile(r"\[\d+\]|\[Q\d+:\d+\]|Source:\s*\S+")


def _parse_sub_questions(raw: str, fallback: str) -> list[str]:
    """Defensively parse the LLM JSON from sub_queries(). Returns [fallback] on any failure."""
    if not isinstance(raw, str) or not raw.strip():
        return [fallback]

    text = raw.strip()
    # handle ```json ... ``` fences
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1)
    else:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        text = match.group(0) if match else None

    if not text:
        return [fallback]

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return [fallback]

    if not isinstance(parsed, dict):
        return [fallback]

    # accept multiple key variants
    questions = None
    for key in ("subQuestions", "sub_questions", "subquestions"):
        if key in parsed:
            questions = parsed[key]
            break

    if not isinstance(questions, list):
        return [fallback]

    cleaned = [q.strip() for q in questions if isinstance(q, str) and q.strip()]
    return cleaned or [fallback]


def _namespace_citations(answer: str, index: int) -> str:
    """Rewrite [1] → [Q{index}:1] so citations from different sub-queries don't collide."""
    return re.sub(r"\[(\d+)\]", lambda m: f"[Q{index}:{m.group(1)}]", answer or "")


async def _run_sub_query(sub_query: str) -> dict:
    """Route and answer one sub-query asynchronously."""
    try:
        decision = await asyncio.to_thread(route_query, sub_query)
        action = decision.get("action")
        reason = decision.get("reason", "")
        fn = routes.get(action)
        if not fn:
            result = f"Unsupported action: {action}"
        elif action in ("OPENAI_QUERY", "10K_DOCUMENT_QUERY"):
            result = await fn(sub_query, action)
        else:
            result = await asyncio.to_thread(fn, sub_query, action)
    except Exception as err:
        action, reason, result = None, "", f"Execution error: {err}"

    return {"query": sub_query, "action": action, "reason": reason, "answer": result}


async def _run_all(questions: list[str]) -> list[dict]:
    return await asyncio.gather(*[_run_sub_query(q) for q in questions])


def _join_parts(parts: list[dict]) -> str:
    """Deterministic fallback if the synthesis LLM drops citations."""
    sections = []
    for i, part in enumerate(parts, 1):
        sections.append(f"### {part['query']}\n{_namespace_citations(part['answer'], i)}")
    return "\n\n".join(sections)


def _compose_answers(user_query: str, parts: list[dict]) -> str:
    if len(parts) == 1:
        return parts[0]["answer"]

    labelled = [
        {**p, "answer": _namespace_citations(p["answer"], i)}
        for i, p in enumerate(parts, 1)
    ]

    blocks = [
        f"Sub-question: {p['query']}\nRoute: {p['action']}\nAnswer:\n{p['answer']}"
        for p in labelled
    ]

    prompt = f"""
Combine the sub-answers below into one coherent response to the original query.
Cover every sub-question. Keep every citation marker exactly as written, including [Q1:1] and Source: URLs.
Do not invent facts that are not in the sub-answers.

Original query: {user_query}

{chr(10).join(blocks)}
"""
    try:
        response = openaiclient.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": prompt}],
        )
        composed = response.choices[0].message.content or ""
    except Exception:
        return _join_parts(labelled)

    # fallback if synthesis dropped all citations
    sources_had_citations = any(_CITATION_RE.search(p["answer"] or "") for p in labelled)
    if sources_had_citations and not _CITATION_RE.search(composed):
        return _join_parts(labelled)

    return composed


def agentic_rag_multi(user_query: str) -> str:
    """
    Split a compound query, route and answer each sub-query concurrently,
    then synthesise one final answer with namespaced citations.

    Args:
        user_query: Possibly compound question.

    Returns:
        Single composed answer covering every sub-question.
    """
    CYAN, GREY, BOLD, RESET = "\033[96m", "\033[90m", "\033[1m", "\033[0m"
    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # Step 1: split
    try:
        questions = _parse_sub_questions(sub_queries(user_query), user_query)
    except Exception as e:
        print(f"{GREY}Split failed ({e}); treating as one query.{RESET}")
        questions = [user_query]

    print(f"{GREY}🔀 Sub-queries ({len(questions)}):{RESET}")
    for i, q in enumerate(questions, 1):
        print(f"{GREY}  {i}. {q}{RESET}")
    print()

    # Step 2: route + answer concurrently
    started = time.perf_counter()
    parts = asyncio.run(_run_all(questions))
    elapsed = time.perf_counter() - started

    for part in parts:
        print(f"{GREY}📍 {part['query']}")
        print(f"   Route: {part['action']} | {part['reason']}{RESET}")
    print(f"{GREY}⏱ Finished in {elapsed:.2f}s (concurrent){RESET}\n")

    # Step 3: synthesise
    result = _compose_answers(user_query, list(parts))
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n{result}\n")
    return result

In [16]:
# Test 1: single question
agentic_rag_multi("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

🔀 Sub-queries (1):
  1. What was Uber's revenue in 2021?

📍 What was Uber's revenue in 2021?
   Route: 10K_DOCUMENT_QUERY | Question is about Uber's financials.
⏱ Finished in 2.05s (concurrent)

🤖 BOT RESPONSE:
Uber's revenue in 2021 was $17.5 billion, which reflects a 57% increase year-over-year [1].



"Uber's revenue in 2021 was $17.5 billion, which reflects a 57% increase year-over-year [1]."

In [17]:
# Test 2: two questions, same route
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

👤 User Query: what was lyft revenue in 2021 and what was uber revenue in 2021

🔀 Sub-queries (2):
  1. What was Lyft's revenue in 2021?
  2. What was Uber's revenue in 2021?

📍 What was Lyft's revenue in 2021?
   Route: 10K_DOCUMENT_QUERY | Question about Lyft 10-K report.
📍 What was Uber's revenue in 2021?
   Route: 10K_DOCUMENT_QUERY | Question about Uber's financials.
⏱ Finished in 3.07s (concurrent)

🤖 BOT RESPONSE:
In 2021, Lyft's revenue was $3,208,323,000 [Q1:2], while Uber's revenue reached $17.5 billion, marking a 57% increase from the previous year [Q2:1].



"In 2021, Lyft's revenue was $3,208,323,000 [Q1:2], while Uber's revenue reached $17.5 billion, marking a 57% increase from the previous year [Q2:1]."

In [18]:
# Test 3: two questions, different routes
agentic_rag_multi("what was uber's 2021 revenue and who won the 2022 FIFA World Cup?")

👤 User Query: what was uber's 2021 revenue and who won the 2022 FIFA World Cup?

🔀 Sub-queries (2):
  1. What was Uber's 2021 revenue?
  2. Who won the 2022 FIFA World Cup?

Getting your response from the internet 🌐 ...
📍 What was Uber's 2021 revenue?
   Route: 10K_DOCUMENT_QUERY | Question about Uber's financials in 2021
📍 Who won the 2022 FIFA World Cup?
   Route: INTERNET_QUERY | User asking about a sports event outcome.
⏱ Finished in 3.99s (concurrent)

🤖 BOT RESPONSE:
Uber's revenue for 2021 was $17.5 billion, representing a 57% increase compared to the previous year [Q1:1]. In the 2022 FIFA World Cup, Argentina emerged victorious, winning their third title by defeating France in the final. The match concluded with a 3-3 draw after extra time, and Argentina triumphed 4-2 in the penalty shootout [Q2:1] [Q2:2] [Q2:3] [Q2:4] [Q2:5].



"Uber's revenue for 2021 was $17.5 billion, representing a 57% increase compared to the previous year [Q1:1]. In the 2022 FIFA World Cup, Argentina emerged victorious, winning their third title by defeating France in the final. The match concluded with a 3-3 draw after extra time, and Argentina triumphed 4-2 in the penalty shootout [Q2:1] [Q2:2] [Q2:3] [Q2:4] [Q2:5]."

---
## Bonus — RBAC-Aware Semantic Cache

**Design choice: Tagged cache (single FAISS index, per-entry source tag)**

All embeddings share one `IndexFlatL2`. Each entry stores the `source` (route action) that produced it. At lookup time, candidates within the distance threshold are filtered to only those whose `source` is in the requesting user's `allowed_sources`. This means:
- A result cached by bob (`10K_DOCUMENT_QUERY`) is never returned to alice, who lacks that permission.
- Users with overlapping permissions (e.g., both can use `OPENAI_QUERY`) share cached entries — more efficient than a fully partitioned design.

**Additional rules:**
- Time-sensitive queries (detected via keyword list) bypass the cache entirely.
- `DENIED` responses and pipeline errors (`Retrieval error`, `Search error`, `Execution error`) are never stored.
- Routing runs **before** the cache lookup: the route decides which source the answer comes from, and the RBAC gate needs it. A HIT therefore saves the retrieval + generation calls, not the routing call.
- Every request is logged to an audit trail.

In [19]:
class RoleAwareSemanticCache:
    """
    Semantic cache with per-entry source tagging.
    A single FAISS IndexFlatL2 holds all embeddings; cache hits are filtered
    by the requesting user's allowed_sources() at lookup time.

    Time-sensitive queries (detected via keyword list) bypass the cache.
    Embeddings are L2-normalised so distance ≈ 1 - cosine_similarity.
    Default threshold 0.2 ≈ cosine similarity ≥ 0.98.
    """

    TIME_SENSITIVE_KEYWORDS = [
        "today", "tonight", "now", "currently", "current",
        "latest", "recent", "recently", "right now", "at the moment",
        "this week", "this month", "this year", "this quarter",
        "yesterday", "tomorrow", "last week", "last month", "last year",
        "upcoming", "live", "breaking", "just happened",
        "stock price", "share price", "weather", "forecast",
        "real-time", "realtime", "news today", "news this week",
    ]

    def __init__(self, threshold: float = 0.2):
        self.threshold = threshold
        self.index = None
        self.entries: list[dict] = []
        self.audit: list[dict] = []

    def is_time_sensitive(self, question: str) -> bool:
        q = (question or "").lower()
        return any(
            re.search(rf"\b{re.escape(kw)}\b", q)
            for kw in self.TIME_SENSITIVE_KEYWORDS
        )

    def _embed(self, question: str) -> np.ndarray:
        vec = np.asarray(get_text_embeddings(question), dtype=np.float32).reshape(1, -1)
        norm = np.linalg.norm(vec, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        return np.ascontiguousarray(vec / norm, dtype=np.float32)

    def check(self, user_id: str, question: str, source: str = None):
        """
        Look up question in the cache, filtered by user's allowed_sources.

        Returns:
            (hit: bool, answer: str | None, embedding: np.ndarray, similarity: float | None)
        """
        embedding = self._embed(question)
        allowed = allowed_sources(user_id)

        if self.index is None or self.index.ntotal == 0:
            return False, None, embedding, None

        distances, indices = self.index.search(embedding, self.index.ntotal)
        for distance, row_id in zip(distances[0], indices[0]):
            if row_id < 0:
                continue
            distance = float(distance)
            if distance > self.threshold:
                break  # sorted by distance; no closer match exists
            entry = self.entries[int(row_id)]
            if entry["source"] not in allowed:
                continue
            if source is not None and entry["source"] != source:
                continue
            return True, entry["answer"], embedding, 1.0 - distance

        return False, None, embedding, None

    def add(self, user_id: str, question: str, answer: str, embedding: np.ndarray, source: str):
        """
        Store an answer tagged with its source route.
        Time-sensitive questions are silently skipped.
        """
        if self.is_time_sensitive(question):
            return
        if not source or source not in allowed_sources(user_id):
            return
        if embedding is None:
            return

        vec = np.asarray(embedding, dtype=np.float32)
        if vec.ndim == 1:
            vec = vec.reshape(1, -1)

        if self.index is None:
            self.index = faiss.IndexFlatL2(vec.shape[1])

        self.index.add(vec)
        self.entries.append({
            "question": question,
            "answer": answer,
            "source": source,
            "user_id": user_id,
        })

    def print_audit(self):
        print(f"\n{'User':<8} {'Role':<18} {'Status':<8} {'Route':<22} {'Latency':>8}  Query")
        print("-" * 90)
        for row in self.audit:
            print(
                f"{row['user']:<8} {str(row['role']):<18} {row['status']:<8} "
                f"{str(row['route']):<22} {row['latency_s']:>7.3f}s  {row['query'][:50]}"
            )

In [20]:
ERROR_PREFIXES = ("Retrieval error", "Search error", "Execution error", "Invalid action", "Unsupported action", "No relevant content", "No results found")


def secure_agentic_rag_cached(user_id: str, user_query: str, cache: RoleAwareSemanticCache) -> dict:
    """
    RBAC-gated agentic RAG with role-aware semantic cache.

    Order: identity → route → RBAC gate → time-sensitivity → cache → pipeline → store.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
    """
    started = time.perf_counter()
    role = USERS.get(user_id)

    def finish(answer: str, status: str, route: str = None) -> dict:
        cache.audit.append({
            "user": user_id,
            "role": role,
            "query": user_query,
            "route": route,
            "status": status,
            "latency_s": round(time.perf_counter() - started, 3),
        })
        return {"answer": answer, "status": status, "role": role}

    # Step 1 — identity
    if role is None:
        return finish(f"🚫 Access denied: unknown user '{user_id}'.", "DENIED")

    # Step 2 — route
    try:
        decision = route_query(user_query)
        action = decision.get("action")
    except Exception as e:
        return finish(f"Routing error: {e}", "DENIED")

    # Step 3 — RBAC gate
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        return finish(
            f"🚫 Access denied: role '{role}' cannot query {source}.",
            "DENIED",
            route=action,
        )

    # Step 4 — time-sensitivity bypass
    if cache.is_time_sensitive(user_query):
        result = _execute_route(user_query, action)
        return finish(result, "MISS", route=action)

    # Step 5 — cache lookup
    hit, answer, embedding, similarity = cache.check(user_id, user_query, source=action)
    if hit:
        return finish(answer, "HIT", route=action)

    # Step 6 — execute pipeline
    try:
        result = _execute_route(user_query, action)
    except Exception as e:
        return finish(f"Execution error: {e}", "MISS", route=action)

    # Step 7 — store (never cache failures)
    if not result.startswith(ERROR_PREFIXES):
        cache.add(user_id, user_query, result, embedding, source=action)
    return finish(result, "MISS", route=action)

## Self-Check

In [21]:
def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"
    assert not r["answer"].startswith(ERROR_PREFIXES), f"pipeline failed: {r['answer']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. near-paraphrase must also be denied, not semantically matched from bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("\n✅ All checks passed — cache is fast and does not leak across roles.")
    cache.print_audit()

In [22]:
run_self_check()


✅ All checks passed — cache is fast and does not leak across roles.

User     Role               Status   Route                   Latency  Query
------------------------------------------------------------------------------------------
bob      finance_analyst    MISS     10K_DOCUMENT_QUERY       2.207s  what was uber revenue in 2021?
bob      finance_analyst    HIT      10K_DOCUMENT_QUERY       0.825s  what was uber revenue in 2021?
alice    engineer           DENIED   10K_DOCUMENT_QUERY       0.857s  what was uber revenue in 2021?
alice    engineer           DENIED   10K_DOCUMENT_QUERY       0.970s  how much revenue did Uber make in 2021?
carol    None               DENIED   None                     0.000s  how do I build an agent with the OpenAI Agents SDK
alice    engineer           MISS     OPENAI_QUERY             4.092s  how do I build an agent with the OpenAI Agents SDK
alice    engineer           HIT      OPENAI_QUERY             0.895s  how do I build an agent with the OpenA